# Phase 3: Typewell features (LGBM v2)

Phase 3 deliverable report. The exploration scratchpad is `03_typewell_explore.ipynb`;
this notebook is the *report*: it states what we tried, the honest result, and what
Phase 4 needs. All runs logged to MLflow experiment `phase3`.

**Phase 3 objective (from `00_phases.md`):** GR slide-and-match against the typewell
beats both Phase 2 baselines on the LB.

**Outcome (stated up front, honestly):** it does **not**. The per-row GR matcher was
shelved (geometry wall — see exploration notebook). The fallback — 4 weak per-well
typewell-derived scalars fed to LGBM v2 — trains and OOFs correctly but lands at
pooled OOF **15.74**, statistically indistinguishable from both carry-forward (15.91)
and LGBM v1 (15.42), all deltas below the ~4-RMSE fold-variance noise floor. The
typewell features carry feature-importance gain but no out-of-sample lift.

This is a clean negative result. It does not mean the typewell is uninformative — the
acid test and affine ceiling below prove the signal is real. It means *this class of
feature* (per-well scalars + a tree) cannot express the signal that exists, which is
**low-frequency drift**, not per-row TVT. That finding is what justifies the Phase 4
architecture.

## What shaped the design (priors carried in)

- **Phase 2:** architecture must be *anchor + small correction* (linear-extrap scored
  ~107). Any feature constant within a well whose distribution shifts train→inference
  sabotages the tree (the 80-RMSE bug).
- **Phase 2:** fold variance ~4.2 RMSE; improvements < 1.5 pooled are below noise.
- **Phase 2:** OOF→LB gap is real and asymmetric — the 3 LB test wells are unusually
  carry-forward-friendly.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from rogii_wellbore.config import MLFLOW_TRACKING_URI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("python :", sys.executable)
print("mlflow :", MLFLOW_TRACKING_URI)

## 1. The signal is real — acid test and affine ceiling

Before reporting the model, establish that the typewell *does* carry TVT information.
Two exploration results (see `03_typewell_explore.ipynb`) make this concrete and
bound what is recoverable.

**Acid test.** In the *known* zone, where true TVT is available, lateral GR plotted
against true TVT tracks the typewell GR-vs-TVT profile bump-for-bump. Across a
30-well stratified sample: **median Pearson r = 0.83, min r = 0.66, 100% of wells
above 0.5.** The single exploration well (`015fe0d2`) hit r = 0.879 over a 494-ft
known-zone TVT span. The GR signal is broadly informative.

**Affine ceiling.** Fitting an oracle 2-parameter affine model
`TVT_pred = anchor + offset + slope·(i − anchor_idx)` directly against *true* eval-zone
TVT (cheating — establishes the ceiling, not an achievable score) gives pooled RMSE
**6.89** across the 30-well sample, vs carry-forward **16.29**. 30/30 wells improve;
max single-well improvement 33.6 RMSE. **There are ~9 RMSE of recoverable affine
drift signal in the data.**

In [ ]:
# Acid test + affine ceiling: headline numbers from the exploration sweep.
# (Reproduced here for the report; full computation in 03_typewell_explore.ipynb.)
acid = pd.Series({"median_r": 0.83, "min_r": 0.66, "frac_above_0.5": 1.00, "n_wells": 30})
ceiling = pd.DataFrame(
    {
        "pooled_rmse": [16.29, 6.89],
        "note": ["carry-forward", "oracle affine fit (offset+slope vs true TVT)"],
    },
    index=["carry_forward", "affine_ceiling"],
)
print("Acid test (known-zone lateral-GR vs typewell-GR correlation):")
print(acid.to_string())
print("\nAffine ceiling (30-well pooled):")
print(ceiling.to_string())
print(f"\nRecoverable affine signal: {16.29 - 6.89:.2f} RMSE below carry-forward.")

## 2. Why the per-row matcher was shelved

Three matcher designs (row-space, TVT-space with constant proxy, TVT-stretched) and a
2-parameter affine-trajectory matcher (maximizing global Pearson r) were attempted in
exploration. All landed at **38–56 RMSE vs CF's 23** on the exploration well.

**Root cause is geometry, not algorithm.** The GR signal carries TVT information only
when the lateral *moves* in TVT. The known zone traverses ~494 ft of TVT — that is why
r=0.88 works there. The eval zone traverses only ~30–40 ft over thousands of rows; the
typewell GR character within that narrow window is not unique enough to disambiguate
`(offset, slope)`. The matcher maximizes correlation correctly, but the correlation
maximum is not at the right TVT.

**Decision:** stop trying to recover per-row TVT. Ship 4 weak per-well scalars and let
LGBM combine them. The matcher's *predicted TVT* is discarded; only its max similarity
score is kept as a (weak, informational) feature.

## 3. LGBM v2 — the 4 typewell features

Per-well scalars, broadcast to every eval row, computed once per well from a *fixed*
reference point (the inference anchor) so they are identically distributed train vs
inference — the Phase 2 lesson applied preventively.

1. `tw_slope_at_anchor` — local typewell dGR/dTVT around the anchor TVT (50-ft window).
2. `gr_delta_eval_anchor` — eval-zone mean lateral GR minus anchor-local lateral GR.
3. `calib_a` — known-zone slope of (lateral GR vs typewell GR at TVT).
4. `matcher_sim` — max Pearson r of the affine matcher (the matcher's TVT is *not* used).

Added on top of v1's 4 features (`dmd`, `dz`, `gr_roll_mean_k`, `gr_roll_std_k`) → 8
features total. Code: `src/rogii_wellbore/features_v2.py`, `models/lgbm_v2.py`,
`oof_v2.py`. Params identical to v1 (`regression_l2`, `lambda_l2=1.0`, `bagging_freq=1`)
so the feature set is the only difference.

### 3.1 The training-window bug (and fix)

The first v2 wiring trained on rows `anchor_idx < row ≤ last_known` — i.e. rows
*inside* the known segment only. But the real eval zone is the masked tail (`row >
last_known`), pure forward extrapolation at large `dmd`. v2 therefore never trained on
a single extrapolation row.

Symptom: the early-stopping val RMSE read ~3–4 while the real eval-zone OOF RMSE was
~12–21 — a 4–7x gap. This is the Phase 2 distribution-shift lesson rotated: not a
leaky constant feature, but the *target's relationship to `dmd`* differing between the
(interpolation) rows trained on and the (extrapolation) rows scored.

**Fix:** mirror v1 — drop the `row_idx ≤ last_known` cap; train on all post-anchor
rows with finite true TVT (which, in training data, includes the tail). After the fix
the val and eval RMSE live in the same range, and the training matrix grows from ~34k
to ~870k rows on the 50-well smoke. This unblocked the full OOF run reported below.

In [ ]:
# Pull the v2 OOF run from MLflow and compare against the Phase 2 references.
runs = mlflow.search_runs(experiment_names=["phase3"])
v2 = runs[runs["tags.mlflow.runName"] == "lgbm_v2"].sort_values("start_time").iloc[-1]

fold_cols = [f"metrics.oof_fold{i}_rmse" for i in range(5)]
v2_folds = v2[fold_cols].to_numpy(dtype=float)

summary = pd.DataFrame(
    {
        "pooled_OOF_RMSE": [15.9099, 15.4199, float(v2["metrics.oof_pooled_rmse"])],
        "delta_vs_CF": [0.0, 15.9099 - 15.4199, 15.9099 - float(v2["metrics.oof_pooled_rmse"])],
    },
    index=["carry_forward", "lgbm_v1", "lgbm_v2"],
).round(4)
print(summary.to_string())
print(f"\nv2 per-fold OOF RMSE: {np.round(v2_folds, 4).tolist()}")
print(f"v2 best iters:        {v2.get('params.best_iters', 'see fold_meta')}")

### 3.2 Result — no out-of-sample lift

Full 770-well OOF (logged to `phase3`):

| model          | pooled OOF RMSE | Δ vs CF        | Δ vs v1        |
|----------------|----------------:|---------------:|---------------:|
| carry-forward  |         15.9099 |             —  |             —  |
| LGBM v1        |         15.4199 |  −0.49 (better) |             —  |
| **LGBM v2**    |     **15.7367** |  −0.17 (better) |  +0.32 (worse) |

**v2 beats CF by 0.17 and loses to v1 by 0.32 — both well under the ~4.2 RMSE
fold-variance noise floor and the 1.5-pooled significance bar.** v2 is statistically
indistinguishable from both. The typewell features did not help.

Two corroborating signals from the run:
- **Best iterations collapsed to 1–9** (vs v1's 14–43). On fold 0 the model
  early-stopped at iteration **1** — a single split captures everything it can learn.
  That is a model finding a weak global structure and nothing well-specific.
- **Per-fold val RMSE swung 14.5 → 21.5 and tr/va inverted across folds** (fold 0:
  tr 18.6 > va 14.5; fold 4: tr 14.8, va 21.5). The per-well features do not transfer
  across well groups — what calibrates on the train wells does not describe the
  holdout wells' eval tails.

In [ ]:
# Feature importance (gain, summed across folds). High gain WITHOUT OOF lift
# is the signature of features the model uses but that do not generalize.
imp = pd.Series(
    {
        "tw_slope_at_anchor": 10_099_375_990,
        "gr_delta_eval_anchor": 9_727_910_882,
        "matcher_sim": 5_359_433_002,
        "dz": 4_563_540_550,
        "calib_a": 4_311_620_800,
        "dmd": 2_163_088_986,
        "gr_roll_mean_k": 416_563_370,
        "gr_roll_std_k": 0,
    }
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [
    "#c0504d"
    if n in {"tw_slope_at_anchor", "gr_delta_eval_anchor", "matcher_sim", "calib_a"}
    else "#4472c4"
    for n in imp.index
]
ax.barh(imp.index[::-1], imp.values[::-1] / 1e9, color=colors[::-1])
ax.set_xlabel("gain (billions, summed across 5 folds)")
ax.set_title("v2 feature importance — typewell features (red) top the table, no OOF lift")
plt.tight_layout()
plt.show()

print(imp.to_string())
print("\nNote: gr_roll_std_k = 0 gain (consistent with the 30-well sandbox).")
print("High-gain-no-lift = model fits per-well structure that does not generalize.")

### 3.3 Submission decision

Phase 3 done-when criterion: beat *both* Phase 2 baselines on LB. v2's OOF (15.74) is
already worse than v1's OOF (15.42), and the OOF→LB gap is asymmetric — spending a
submission on a model CV says is no better than the one already submitted is low value.

**Decision: do not submit v2.** Record the negative OOF result, preserve the
submission budget. (If the LB datapoint is wanted for completeness, one submission is
defensible, but it is not expected to change the conclusion.)

## 4. Conclusion

**Typewell-derived per-well scalar features do not improve out-of-sample TVT
prediction through a tree model.** Stated precisely:

1. The typewell signal is **real** — known-zone GR correlation median r = 0.83, and an
   oracle affine fit recovers ~9 RMSE below carry-forward. The information exists.
2. That information is **low-frequency drift** (offset + slope across the eval tail),
   **not per-row TVT** — the eval zone's ~30-ft TVT traverse is too narrow for per-row
   GR matching, and no matcher design overcame this geometry wall.
3. **Per-well scalars + LGBM cannot express drift.** A tree predicts a constant per
   leaf; forward-extrapolating a per-well slope is structurally what it does worst.
   The features show high gain but zero OOF lift, best-iterations collapse to 1–9, and
   the per-well constants fail to transfer across folds.

This corroborates and sharpens the Phase 2 conclusion: the architecture must be
*anchor + small correction*, and the correction is a **drift trend that must be
extrapolated**, which a per-row tree on per-well scalars cannot do.

**Phase 3 closed honestly. Tag `v0.3-typewell`.** Phase tracker: Phase 3 done-when
("beats both Phase 2 baselines on LB") is **not met**; recorded as a negative result
with the diagnosis above, which redirects Phase 4 rather than blocking it.

## 5. What Phase 4 needs

The affine ceiling (6.89 vs CF 16.29) says ~9 RMSE of drift signal is sitting unclaimed.
The job of Phase 4 is to capture drift that a tree cannot. Concretely:

**Architecture — a sequence model that learns anchor + extrapolated correction
end-to-end.** A 1D-CNN or small transformer over a windowed view of the lateral
(GR sequence + `dz`, `dmd` positional features), predicting the residual-from-anchor
*as a function of distance down the eval tail*. Unlike the tree, it can represent a
continuous drift slope and carry it forward. (Phase tracker Phase 4: "1D-CNN/Transformer
with delta-TVT + masked input beats Phase 3 on OOF and LB.")

**Inputs to carry over (these are validated, reuse them):**
- Residual-from-anchor target; inference anchor = last known row. *(Phase 2)*
- Multi-anchor training with `fracs (0.95, 0.90, 0.85, 0.80)`, **and** — critically —
  training rows must include the extrapolation tail (the v2 window bug fix). *(Phase 3)*
- Causal/leakage-safe features only; uniform 1.0-ft MD grid (no resampling needed,
  Phase 1 Q4); per-well GR handling (interpolate short NaN runs, Phase 1 Q2).
- The typewell as an *auxiliary input sequence*, not a per-well scalar — let the model
  attend to the GR-vs-TVT profile directly rather than pre-summarizing it. The
  exploration shows the matchable signal is there; a learned model may use it where a
  hand-built matcher could not.

**De-risking step before committing to Phase 4 (recommended, ~1 hour):** run a plain
per-well affine extrapolation — slope fit on the known-zone tail only (no peeking),
extrapolated forward — through the existing OOF harness. The affine *ceiling* (oracle
slope) is measured; the *achievable causal* version is not. If causal affine beats CF
on OOF, Phase 4 has a concrete target and strong prior it will pay off; if it doesn't,
the drift is not cleanly recoverable causally and Phase 4's framing changes. Either
outcome de-risks the most expensive phase.

**Evaluation discipline (carried from Phase 2):**
- Well-grouped pooled OOF is the headline; fold variance ~4.2 RMSE.
- Improvements < 1.5 pooled require **multi-seed runs** before any claim.
- The 3 LB test wells are unusually CF-friendly; OOF→LB is asymmetric. Check
  pad-grouped CV before trusting public LB.